# <font color="#418FDE" size="6.5" uppercase>**Dichtes PyTorch-Netz**</font>

>Last update: 20260825.
    
By the end of this Lecture, you will be able to:
- Definieren ein kleines dichtes PyTorch-Modell mit nn.Module und forward. 
- Trainieren das Modell mit Verlustfunktion, Optimierer und eigener Schleife. 
- Speichern, laden und prüfen Vorhersagen mit state_dict. 


## **1. Modell definieren**

### **1.1. nn Module verstehen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_17/Lecture_B/image_01_01.jpg?v=1787661688" width="250">



>* nn.Module strukturiert dichte PyTorch-Modelle.
>* Parameterverwaltung und Training werden automatisch unterstützt.

>* Modellaufbau und Berechnung klar trennen
>* Schichten registrieren lernbare Parameter automatisch

>* Modell als wiederverwendbare Einheit verstehen
>* Struktur macht Schichten und Datenfluss nachvollziehbar



In [ ]:
#@title Python-Code - nn Module verstehen

# Dieses Beispiel zeigt ein kleines dichtes PyTorch-Modell.
# nn.Module registriert Schichten und lernbare Parameter automatisch.
# Die Ausgabe zeigt Formen, Parameter und eine Vorhersage.

import torch
from torch import nn

# Eine feste Startzahl macht die Beispielwerte reproduzierbar.
torch.manual_seed(42)

# Diese Klasse beschreibt den Bauplan des Netzes.
class DenseModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(3, 4)

        self.activation = nn.ReLU()
        self.output = nn.Linear(4, 1)

    def forward(self, x):
        x = self.hidden(x)
        x = self.activation(x)
        return self.output(x)

# Das Modellobjekt enthält nun registrierte Schichten.
model = DenseModel()

# Zwei Beispiele mit jeweils drei Merkmalen.
features = torch.tensor(
    [[0.2, 0.5, 0.1], [0.9, 0.1, 0.4]],
    dtype=torch.float32,
)

# Eine einfache Prüfung verhindert unpassende Eingabeformen.
if features.shape[1] != 3:
    raise ValueError("Die Eingabe braucht genau drei Merkmale.")

# Der Aufruf des Modells nutzt automatisch forward.
predictions = model(features)

# named_parameters zeigt, was PyTorch trainieren kann.
parameter_count = sum(parameter.numel() for parameter in model.parameters())

print("Eingabeform:", tuple(features.shape))
print("Ausgabeform:", tuple(predictions.shape))
print("Trainierbare Parameter:", parameter_count)
print("Registrierte Parametergruppen:", len(list(model.named_parameters())))
print("Erste Vorhersage:", round(float(predictions[0, 0]), 4))



### **1.2. forward Methode**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_17/Lecture_B/image_01_02.jpg?v=1787661691" width="250">



>* forward legt den Datenweg im Modell fest
>* Schichten verwandeln Eingaben schrittweise in Vorhersagen

>* Daten fließen schichtweise durch lineare Transformationen
>* Aktivierungen verbinden Schritte zu komplexeren Mustern

>* forward berechnet Ausgaben, aktualisiert keine Gewichte
>* Beschreibt klar den Informationsfluss im Modell



In [ ]:
#@title Python-Code - forward Methode

# Dieses Beispiel zeigt die forward Methode.
# Ein dichtes Netz verarbeitet kleine Eingaben.
# Die Ausgabe macht den Datenfluss sichtbar.

import torch

# Feste Zufallswerte machen das Beispiel reproduzierbar.
torch.manual_seed(42)

# Diese Klasse beschreibt ein kleines dichtes Modell.
class DenseModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = torch.nn.Linear(3, 4)

        self.activation = torch.nn.ReLU()
        self.output = torch.nn.Linear(4, 1)

    # Die forward Methode legt die Reihenfolge fest.
    def forward(self, x):
        x = self.hidden(x)
        x = self.activation(x)

        x = self.output(x)
        return x

# Zwei Beispiele mit jeweils drei Merkmalen.
features = torch.tensor(
    [[70.0, 20.0, 8.0], [45.0, 60.0, 6.0]],
    dtype=torch.float32,
)

# Die Form muss zur ersten linearen Schicht passen.
if features.shape[1] != 3:
    raise ValueError("Die Eingaben brauchen genau drei Merkmale.")

# Ein Modellaufruf verwendet automatisch forward.
model = DenseModel()
predictions = model(features)

# Die Ausgabe hat eine Vorhersage pro Eingabezeile.
print(f"Eingabeform: {tuple(features.shape)}")
print(f"Ausgabeform: {tuple(predictions.shape)}")
print(f"Erste Vorhersage: {predictions[0, 0].item():.2f}")
print("model(features) ruft intern die forward Methode auf.")



### **1.3. Ausgaben konfigurieren**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_17/Lecture_B/image_01_03.jpg?v=1787661689" width="250">



>* Ausgabeform passend zur Aufgabe wählen
>* Letzte Schicht, Verlust und Interpretation abstimmen

>* Logits sind Rohwerte, keine Wahrscheinlichkeiten
>* Auswertung macht daraus verständliche Klassenentscheidungen

>* Ausgabeform muss Stapel und Ziele abbilden
>* Letzte Schicht passend zur Verlustfunktion wählen



In [ ]:
#@title Python-Code - Ausgaben konfigurieren

# Dieses Beispiel zeigt passende Ausgabeschichten.
# Die letzte Schicht bestimmt die Vorhersageform.
# Wir prüfen Logits für verschiedene Aufgaben.

import torch
from torch import nn

# Eine feste Startzahl macht die Gewichte reproduzierbar.
torch.manual_seed(42)

# Vier Beispiele mit jeweils fünf Eingabemerkmalen.
batch_size = 4
input_features = 5
x = torch.randn(batch_size, input_features)

# Dieses dichte Modell gibt genau eine Zielzahl aus.
regression_model = nn.Sequential(
    nn.Linear(input_features, 8),
    nn.ReLU(),
    nn.Linear(8, 1),
)

# Dieses dichte Modell gibt drei Klassenwerte aus.
classification_model = nn.Sequential(
    nn.Linear(input_features, 8),
    nn.ReLU(),
    nn.Linear(8, 3),
)

# Die Ausgaben sind Rohwerte, also noch keine Wahrscheinlichkeiten.
regression_output = regression_model(x)
class_logits = classification_model(x)
class_probabilities = torch.softmax(class_logits, dim=1)

# Eine einfache Prüfung verhindert unpassende Ausgabeformen.
if regression_output.shape != (batch_size, 1):
    raise ValueError("Die Regressionsausgabe hat die falsche Form.")

# Auch die Klassenausgabe muss zur Aufgabenstellung passen.
if class_logits.shape != (batch_size, 3):
    raise ValueError("Die Klassifikationsausgabe hat die falsche Form.")

# Die sichtbaren Ergebnisse zeigen Stapelgröße und Zielstruktur.
print(f"Eingabeform: {tuple(x.shape)}")
print(f"Regression: {tuple(regression_output.shape)} = eine Zahl pro Beispiel")
print(f"Klassifikation: {tuple(class_logits.shape)} = drei Logits pro Beispiel")
print("Erste Logits: " + str(class_logits[0].detach().round(decimals=3).tolist()))
print("Erste Wahrscheinlichkeiten: " + str(class_probabilities[0].detach().round(decimals=3).tolist()))



## **2. Training schreiben**

### **2.1. Verlustfunktion auswählen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_17/Lecture_B/image_02_01.jpg?v=1787661682" width="250">



>* Verlust misst Abweichung und steuert Lernen
>* Sie muss zur Aufgabe passen

>* Regression bewertet Abstände zu Zielwerten
>* Verlustwahl bestimmt Umgang mit Fehlern

>* Verlustfunktion an Klassifikationsart anpassen
>* Ausgabeform vor dem Training prüfen



In [ ]:
#@title Python-Code - Verlustfunktion auswählen

# Dieses Beispiel vergleicht passende Verlustfunktionen.
# Regression und Klassifikation brauchen unterschiedliche Signale.
# Die Ausgaben zeigen typische Verlustwerte.

import torch
import torch.nn as nn

# Wir verwenden feste Werte für reproduzierbare Ergebnisse.
torch.manual_seed(42)

# Regressionsausgaben sind direkte Zahlenwerte.
regression_predictions = torch.tensor([[2.5], [3.0], [4.2]])
regression_targets = torch.tensor([[3.0], [2.0], [4.0]])

# MSELoss bestraft größere Zahlenfehler besonders stark.
mse_loss = nn.MSELoss()
regression_loss = mse_loss(regression_predictions, regression_targets)

# Klassifikationsausgaben sind rohe Klassenwerte, sogenannte Logits.
class_logits = torch.tensor([[2.0, 0.5, -1.0], [0.1, 1.8, 0.2]])
class_targets = torch.tensor([0, 1])

# CrossEntropyLoss erwartet Logits und Klassenindizes.
cross_entropy_loss = nn.CrossEntropyLoss()
classification_loss = cross_entropy_loss(class_logits, class_targets)

# Die Form der Zielwerte muss zur Verlustfunktion passen.
print("Regression: Vorhersageform", tuple(regression_predictions.shape))
print("Regression: Zielform", tuple(regression_targets.shape))
print("Regression: MSELoss", round(regression_loss.item(), 4))
print("Klassifikation: Logitform", tuple(class_logits.shape))
print("Klassifikation: Zielform", tuple(class_targets.shape))
print("Klassifikation: CrossEntropyLoss", round(classification_loss.item(), 4))
print("Merke: Verlustfunktion nach Aufgabe und Ausgabeform wählen.")



### **2.2. Optimierer auswählen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_17/Lecture_B/image_02_02.jpg?v=1787661684" width="250">



>* Optimierer wandeln Gradienten in Parameteränderungen um
>* PyTorch verbindet sie mit trainierbaren Modellwerten

>* Optimierer passend zur Trainingssituation wählen
>* Adam passt Schritte adaptiv an

>* Lernrate steuert Größe der Trainingsschritte
>* Training beobachten und Optimierer passend abstimmen



In [ ]:
#@title Python-Code - Optimierer auswählen

# Dieses Beispiel vergleicht zwei Optimierer beim Training.
# Die Lernrate steuert die Größe der Updates.
# Der Verlustverlauf zeigt den Unterschied sichtbar.

import numpy as np
import matplotlib.pyplot as plt
import torch

# Feste Zufallswerte machen das Ergebnis reproduzierbar.
torch.manual_seed(42)
rng = np.random.default_rng(42)

# Wir erzeugen kleine Regressionsdaten mit bekanntem Muster.
x_values = rng.uniform(-2.0, 2.0, size=(80, 1)).astype(np.float32)
noise = rng.normal(0.0, 0.25, size=(80, 1)).astype(np.float32)

y_values = (3.0 * x_values - 1.0 + noise).astype(np.float32)
inputs = torch.tensor(x_values)
targets = torch.tensor(y_values)

# Diese Prüfung verhindert unpassende Trainingsformen.
if inputs.shape != targets.shape:
    raise ValueError("Eingaben und Ziele müssen dieselbe Form haben.")

# Ein kleines dichtes Modell reicht für diese Aufgabe.
class TinyDenseModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = torch.nn.Linear(1, 1)

    def forward(self, x):
        return self.layer(x)

# Diese Funktion trainiert dasselbe Modell mit einem Optimierer.
def train_with_optimizer(optimizer_name):
    torch.manual_seed(42)
    model = TinyDenseModel()
    loss_function = torch.nn.MSELoss()

    if optimizer_name == "SGD":
        optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
    else:
        optimizer = torch.optim.Adam(model.parameters(), lr=0.05)

    losses = []
    for epoch in range(40):
        predictions = model(inputs)
        loss = loss_function(predictions, targets)
        optimizer.zero_grad()

        loss.backward()
        optimizer.step()
        losses.append(float(loss.detach()))

    return losses

# Beide Optimierer sehen dieselben Daten und dieselbe Startlogik.
sgd_losses = train_with_optimizer("SGD")
adam_losses = train_with_optimizer("Adam")

# Kurze Zahlen zeigen den Effekt ohne lange Trainingsausgabe.
print("PyTorch-Version:", torch.__version__)
print("Startverlust SGD:", round(sgd_losses[0], 3))
print("Endverlust SGD:", round(sgd_losses[-1], 3))
print("Endverlust Adam:", round(adam_losses[-1], 3))

# Der Plot macht die Optimiererwahl direkt vergleichbar.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sgd_losses, label="SGD")
ax.plot(adam_losses, label="Adam")

ax.set_title("Verlustverlauf mit zwei Optimierern")
ax.set_xlabel("Epoche")
ax.set_ylabel("Mittlerer quadratischer Fehler")
ax.legend()
plt.show()



### **2.3. Train und Eval**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_17/Lecture_B/image_02_03.jpg?v=1787661686" width="250">



>* Trainingsmodus lässt das Modell aus Beispielen lernen
>* Wiederholte Schritte verbessern die Modellparameter systematisch

>* Eval prüft Vorhersagen ohne weiteres Lernen
>* Trainings- und Prüfphasen sauber trennen

>* Trainingsschleife steuert Lernen und Auswertung
>* Train und Eval trennen Kontrolle sauber



In [ ]:
#@title Python-Code - Train und Eval

# Dieses Beispiel trainiert ein kleines dichtes Netz.
# Train und Eval werden bewusst getrennt.
# Die Verluste zeigen den Lernfortschritt sichtbar.

import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt

# Feste Startwerte machen das Ergebnis reproduzierbar.
np.random.seed(42)
torch.manual_seed(42)

# Wir erzeugen kleine Regressionsdaten ohne Download.
x_values = np.linspace(-2.0, 2.0, 120, dtype=np.float32).reshape(-1, 1)
y_values = 3.0 * x_values + 1.0

# Ein wenig Rauschen macht die Aufgabe realistischer.
noise = np.random.normal(0.0, 0.25, size=y_values.shape).astype(np.float32)
y_values = y_values + noise

# Trainingsdaten und Validierungsdaten bleiben getrennt.
x_train = torch.tensor(x_values[:90])
y_train = torch.tensor(y_values[:90])

x_eval = torch.tensor(x_values[90:])
y_eval = torch.tensor(y_values[90:])

# Diese Prüfung verhindert unpassende Eingabeformen.
if x_train.shape[1] != 1 or y_train.shape[1] != 1:
    raise ValueError("Die Daten müssen genau eine Eingabe und ein Ziel haben.")

# Ein dichtes Modell erbt von nn.Module.
class DenseRegressionNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(nn.Linear(1, 8), nn.ReLU(), nn.Linear(8, 1))

    def forward(self, x):
        return self.layers(x)

# Verlustfunktion und Optimierer steuern das Lernen.
model = DenseRegressionNet()
loss_function = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

train_losses = []
eval_losses = []

# Jede Epoche hat eine Trainingsphase und eine Evalphase.
for epoch in range(80):
    model.train()
    predictions = model(x_train)
    train_loss = loss_function(predictions, y_train)

    optimizer.zero_grad()
    train_loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        eval_predictions = model(x_eval)
        eval_loss = loss_function(eval_predictions, y_eval)

    train_losses.append(train_loss.item())
    eval_losses.append(eval_loss.item())

# Kurze Ausgaben vergleichen Anfang und Ende.
print(f"PyTorch-Version: {torch.__version__}")
print(f"Train-Verlust Start: {train_losses[0]:.3f}")
print(f"Train-Verlust Ende: {train_losses[-1]:.3f}")
print(f"Eval-Verlust Ende: {eval_losses[-1]:.3f}")

# Die Kurven zeigen Lernen und Auswertung getrennt.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(train_losses, label="Train-Verlust")
ax.plot(eval_losses, label="Eval-Verlust")
ax.set_title("Train und Eval in einer eigenen Schleife")
ax.set_xlabel("Epoche")
ax.set_ylabel("Mittlerer quadratischer Fehler")
ax.legend()
plt.show()



## **3. Reproduzierbar speichern**

### **3.1. Metriken visualisieren**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_17/Lecture_B/image_03_01.jpg?v=1787661693" width="250">



>* Trainingskurven zeigen Entwicklung statt Einzelwerte.
>* Sie helfen, gute Speicherpunkte zu erkennen.

>* Mehrere passende Metriken gemeinsam betrachten
>* Kurvenmuster für Speicherzeitpunkt nutzen

>* Metriken begründen den gespeicherten Modellzustand
>* Visualisierungen sichern nachvollziehbare Modellbewertung



In [ ]:
#@title Python-Code - Metriken visualisieren

# Wir visualisieren Metriken eines gespeicherten PyTorch-Modells.
# Das Beispiel verbindet Training, state_dict und Validierung.
# Die Kurven zeigen den besten Speicherzeitpunkt.

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn

# Feste Startwerte machen das Training reproduzierbar.
torch.manual_seed(42)
rng = np.random.default_rng(42)

# Kleine synthetische Daten halten das Beispiel übersichtlich.
x_values = rng.uniform(-2.0, 2.0, size=(120, 1)).astype(np.float32)
noise = rng.normal(0.0, 0.25, size=(120, 1)).astype(np.float32)

y_values = (2.0 * x_values + 0.5 + noise).astype(np.float32)
indices = rng.permutation(len(x_values))

# Training und Validierung werden sauber getrennt.
train_indices = indices[:90]
valid_indices = indices[90:]

x_train = torch.tensor(x_values[train_indices])
y_train = torch.tensor(y_values[train_indices])

x_valid = torch.tensor(x_values[valid_indices])
y_valid = torch.tensor(y_values[valid_indices])

# Ein kleines dichtes Netz reicht für die Demonstration.
class DenseRegressionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(nn.Linear(1, 8), nn.ReLU(), nn.Linear(8, 1))

    def forward(self, inputs):
        return self.layers(inputs)

# Verlustfunktion und Optimierer steuern die Lernschritte.
model = DenseRegressionModel()
loss_function = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

train_losses = []
valid_losses = []
best_valid_loss = float("inf")
best_state_dict = None

# Jede Epoche liefert Metriken für die spätere Kurve.
for epoch in range(40):
    model.train()
    optimizer.zero_grad()
    train_predictions = model(x_train)
    train_loss = loss_function(train_predictions, y_train)
    train_loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        valid_predictions = model(x_valid)
        valid_loss = loss_function(valid_predictions, y_valid)

    train_losses.append(float(train_loss.item()))
    valid_losses.append(float(valid_loss.item()))

    if valid_losses[-1] < best_valid_loss:
        best_valid_loss = valid_losses[-1]
        best_state_dict = {
            key: value.detach().clone() for key, value in model.state_dict().items()
        }

# Eine einfache Prüfung schützt vor einem fehlenden Speicherzustand.
if best_state_dict is None:
    raise RuntimeError("Kein Modellzustand wurde gespeichert.")

best_epoch = int(np.argmin(valid_losses)) + 1
loaded_model = DenseRegressionModel()
loaded_model.load_state_dict(best_state_dict)

# Das geladene Modell wird wie ein gespeichertes Modell geprüft.
loaded_model.eval()
with torch.no_grad():
    loaded_valid_loss = loss_function(loaded_model(x_valid), y_valid).item()

print(f"PyTorch-Version: {torch.__version__}")
print(f"Beste Validierungsepoche: {best_epoch}")
print(f"Geladener Validierungsverlust: {loaded_valid_loss:.4f}")

# Eine einzige Grafik macht den Trainingsverlauf sichtbar.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, 41), train_losses, label="Trainingsverlust")
ax.plot(range(1, 41), valid_losses, label="Validierungsverlust")

ax.scatter(best_epoch, best_valid_loss, color="red", label="gespeicherter Zustand")
ax.set_title("Metriken zum gespeicherten state_dict")
ax.set_xlabel("Epoche")
ax.set_ylabel("Mittlerer quadratischer Fehler")
ax.legend()

plt.show()



### **3.2. Gewichte sicher speichern**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_17/Lecture_B/image_03_02.jpg?v=1787661695" width="250">



>* state_dict speichert den gelernten Modellzustand
>* Gewichte später zuverlässig laden und prüfen

>* Modellgewichte brauchen klaren Projektkontext
>* Saubere Namen verhindern gefährliche Verwechslungen

>* Vorhersagen vor und nach dem Laden vergleichen
>* state_dict sichert reproduzierbare Modellnutzung



In [ ]:
#@title Python-Code - Gewichte sicher speichern

# Dieses Beispiel speichert PyTorch-Gewichte sicher.
# state_dict enthält nur den gelernten Modellzustand.
# Geladene Vorhersagen sollen identisch bleiben.

import torch
from torch import nn

# Feste Zufallszahlen machen Training und Prüfung reproduzierbar.
torch.manual_seed(42)

# Kleine Trainingsdaten zeigen eine einfache lineare Beziehung.
features = torch.tensor([[0.0], [1.0], [2.0], [3.0]])
targets = torch.tensor([[1.0], [3.0], [5.0], [7.0]])

# Ein kleines dichtes Netz passt zur gespeicherten Architektur.
class TinyDenseNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(nn.Linear(1, 8), nn.ReLU(), nn.Linear(8, 1))

    def forward(self, x):
        return self.layers(x)

# Modell, Verlustfunktion und Optimierer bilden die Trainingsschleife.
model = TinyDenseNet()
loss_function = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

# Die Schleife trainiert nur kurz für ein schnelles Beispiel.
for epoch in range(80):
    predictions = model(features)
    loss = loss_function(predictions, targets)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# eval deaktiviert trainingsabhängiges Verhalten vor der Prüfung.
model.eval()
check_input = torch.tensor([[4.0]])

# no_grad verhindert unnötige Gradienten beim Vorhersagen.
with torch.no_grad():
    prediction_before = model(check_input)

# state_dict wird hier im Speicher kopiert, ohne Datei zu schreiben.
saved_state = {}
for name, tensor in model.state_dict().items():
    saved_state[name] = tensor.detach().clone()

# Ein neues Modell braucht dieselbe Architektur vor dem Laden.
loaded_model = TinyDenseNet()
loaded_model.load_state_dict(saved_state)
loaded_model.eval()

# Dieselbe Eingabe prüft, ob die Gewichte korrekt geladen wurden.
with torch.no_grad():
    prediction_after = loaded_model(check_input)

# allclose erlaubt winzige numerische Unterschiede beim Vergleich.
same_prediction = torch.allclose(prediction_before, prediction_after)

# Die Ausgabe zeigt den Kern der sicheren Gewichtsspeicherung.
print(f"Vor dem Laden: {prediction_before.item():.3f}")
print(f"Nach dem Laden: {prediction_after.item():.3f}")
print(f"Vorhersagen stimmen überein: {same_prediction}")



### **3.3. Mini Projekt speichern**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_17/Lecture_B/image_03_03.jpg?v=1787661697" width="250">



>* state_dict sichert gelernte Modellparameter zuverlässig
>* Gespeicherte Modelle vermeiden erneutes Training

>* Gleiche Architektur vor dem Laden erstellen
>* Datenvorbereitung und Eingabereihenfolge dokumentieren

>* Geladene Modelle mit bekannten Beispielen prüfen
>* Speichern, Laden und Vorverarbeitung nachvollziehbar halten



In [ ]:
#@title Python-Code - Mini Projekt speichern

# Dieses Mini Projekt speichert ein dichtes PyTorch Modell.
# state_dict bewahrt gelernte Parameter reproduzierbar auf.
# Geladene Vorhersagen sollen exakt übereinstimmen.

import torch
import matplotlib.pyplot as plt

# Feste Seeds machen Training und Prüfung nachvollziehbar.
torch.manual_seed(42)

# Kleine Trainingsdaten beschreiben eine einfache lineare Beziehung.
features = torch.tensor([[20.0], [35.0], [50.0], [65.0], [80.0]])
targets = torch.tensor([[60.0], [105.0], [150.0], [195.0], [240.0]])

# Skalierung hält die Zahlen für das Netz angenehm klein.
features_scaled = features / 100.0
targets_scaled = targets / 300.0

# Die Architektur muss beim Laden identisch wieder erstellt werden.
class DensePriceModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = torch.nn.Sequential(
            torch.nn.Linear(1, 8),
            torch.nn.ReLU(),
            torch.nn.Linear(8, 1),
        )

    def forward(self, x):
        return self.layers(x)

# Ein kleines Modell lernt die Trainingsbeispiele.
model = DensePriceModel()
loss_function = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)

# Die Trainingsschleife bleibt kurz und vollständig sichtbar.
for epoch in range(80):
    predictions = model(features_scaled)
    loss = loss_function(predictions, targets_scaled)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# Referenzvorhersagen werden vor dem Speichern notiert.
model.eval()
with torch.no_grad():
    reference_predictions = model(features_scaled) * 300.0

# state_dict wird hier im Speicher statt als Datei gesichert.
saved_state = {}
for name, tensor in model.state_dict().items():
    saved_state[name] = tensor.detach().clone()

# Ein neues Modell erhält dieselbe Architektur und die Gewichte.
loaded_model = DensePriceModel()
loaded_model.load_state_dict(saved_state)
loaded_model.eval()

# Nach dem Laden prüfen wir dieselben Beispiele erneut.
with torch.no_grad():
    loaded_predictions = loaded_model(features_scaled) * 300.0

# Die maximale Abweichung zeigt, ob das Laden korrekt war.
max_difference = torch.max(torch.abs(reference_predictions - loaded_predictions))
print(f"Trainingsverlust: {loss.item():.6f}")
print(f"Gespeicherte Parametergruppen: {len(saved_state)}")
print(f"Maximale Abweichung nach dem Laden: {max_difference.item():.8f}")
print(f"Prüfung bestanden: {torch.allclose(reference_predictions, loaded_predictions)}")

# Die Grafik vergleicht Zielwerte und geladene Vorhersagen.
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(features.numpy(), targets.numpy(), label="Zielwerte")
ax.plot(features.numpy(), loaded_predictions.detach().numpy(), label="Geladenes Modell")
ax.set_title("Vorhersagen nach state_dict Laden")
ax.set_xlabel("Wohnfläche in Quadratmetern")
ax.set_ylabel("Preis in Tausend Euro")
ax.legend()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Dichtes PyTorch-Netz**</font>


In this lecture, you learned to:
- Definieren ein kleines dichtes PyTorch-Modell mit nn.Module und forward. 
- Trainieren das Modell mit Verlustfunktion, Optimierer und eigener Schleife. 
- Speichern, laden und prüfen Vorhersagen mit state_dict. 

In the next Module (Module 18), we will go over 'PyTorch CNNs'